# Download historical prices

This notebook locates the project automatically and writes adjusted daily closes to `data/processed/prices.parquet`. The panel dimensions vary with configuration and provider coverage. The coverage table uses ISO dates and observation counts; the price preview shows the latest five dates for at most twelve symbols. A dash is a missing quote, not a zero price.


In [ ]:
import sys
from pathlib import Path
from IPython.display import display

candidates = [Path.cwd(), *Path.cwd().parents]
project_root = next((path for path in candidates if (path / 'src' / 'trading').is_dir()), None)
if project_root is None:
    raise RuntimeError('Could not locate the project root containing src/trading')
sys.path.insert(0, str(project_root / 'src'))

from trading.config import load_config
from trading.data import configured_symbols, coverage_report, download_prices
from trading.open_data import merge_alpha_vantage_prices

config = load_config(project_root / 'config.yaml')
symbols = configured_symbols(config.values)
prices = download_prices(symbols, config['start_date'], config['end_date'], project_root / 'data' / 'processed' / 'prices.parquet')
alpha_symbols = config.values.get('alpha_vantage_symbols', [])
if alpha_symbols:
    if not __import__('os').environ.get('ALPHAVANTAGE_API_KEY'):
        raise RuntimeError('alpha_vantage_symbols is configured but ALPHAVANTAGE_API_KEY is missing')
    prices = merge_alpha_vantage_prices(prices, alpha_symbols)
    prices.to_parquet(project_root / 'data' / 'processed' / 'prices.parquet')
print(f'{prices.shape[0]:,} dates x {prices.shape[1]:,} symbols')
print(f'Date range: {prices.index.min().date()} to {prices.index.max().date()}')
coverage_preview = coverage_report(prices).sort_values('observations').head(20)
display(coverage_preview.style.format({
    'first_date': '{:%Y-%m-%d}', 'last_date': '{:%Y-%m-%d}', 'observations': '{:,.0f}'
}, na_rep='—'))
price_preview = prices.tail().loc[:, prices.columns[:12]]
display(price_preview.style.format('{:,.2f}', na_rep='—').format_index('{:%Y-%m-%d}'))